# Linderman Imágenes (ComfyUI gratis en Kaggle)

**Antes de darle Run All**, en el panel de la derecha (Settings / Session options):
1. **Accelerator → GPU T4 x2**
2. **Internet → On** (si no aparece, falta verificar su teléfono en kaggle.com/settings)

Luego arriba: **Run All**. Tarda unos 8-12 minutos la primera vez (baja los modelos).
Al final sale un botón verde **ABRIR GENERADOR**: ese link sirve en la compu y en el teléfono.

- Deje esta pestaña abierta mientras use el generador. Si la cierra, Kaggle apaga la máquina.
- Las imágenes se guardan en la carpeta `imagenes` (panel derecho → Output). Bájelas antes de apagar.
- No comparta el link: el que lo tenga puede usar su máquina.


In [ ]:
# ===== AJUSTES: lo único que se toca =====
MODELO_REALISTA = True    # RealVisXL V5: fotos que parecen reales (~7 GB)
MODELO_ANIME    = True    # NoobAI-XL 1.1: anime y personajes (~7 GB)
MODELO_FLUX     = False   # FLUX.1 dev: el más realista, pero 3-5 veces más lento (~18 GB)

# Modelos o estilos (LoRA) de Civitai: el número de VERSIÓN que sale en el link
# civitai.com/models/XXXX?modelVersionId=NUMERO  ->  ejemplo: CIVITAI = [1759168, 2071281]
# Necesita el secreto CIVITAI_TOKEN (Add-ons → Secrets). Solo modelos SDXL, Pony o Illustrious.
CIVITAI = []


In [ ]:
APP_HTML = r'''<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Linderman Imágenes</title>
<style>
:root{--bg:#111316;--panel:#1b1e23;--borde:#2c3138;--texto:#e9ecef;--suave:#9aa3ad;--acento:#3fb27f;--error:#e5534b}
*{box-sizing:border-box}
body{margin:0;background:var(--bg);color:var(--texto);font:16px/1.45 system-ui,-apple-system,Segoe UI,Roboto,sans-serif}
main{max-width:980px;margin:0 auto;padding:16px}
h1{font-size:20px;margin:4px 0 14px}
.panel{background:var(--panel);border:1px solid var(--borde);border-radius:12px;padding:14px;margin-bottom:14px}
label{display:block;font-size:13px;color:var(--suave);margin:10px 0 4px}
textarea,select,input{width:100%;background:#0d0f12;color:var(--texto);border:1px solid var(--borde);border-radius:8px;padding:10px;font:inherit}
textarea{min-height:96px;resize:vertical}
.fila{display:grid;grid-template-columns:1fr 1fr 1fr;gap:10px}
@media(max-width:640px){.fila{grid-template-columns:1fr}}
button{border:0;border-radius:10px;padding:13px 18px;font:600 16px system-ui,sans-serif;cursor:pointer}
#crear{background:var(--acento);color:#06130d;width:100%;margin-top:14px}
#crear:disabled{opacity:.5;cursor:wait}
#cancelar{background:#2c3138;color:var(--texto);margin-top:8px;width:100%;display:none}
details summary{cursor:pointer;color:var(--suave);font-size:14px;margin-top:12px}
#estado{min-height:22px;font-size:14px;color:var(--suave);margin-top:10px}
#estado.error{color:var(--error)}
.barra{height:6px;background:#0d0f12;border-radius:4px;overflow:hidden;margin-top:6px;display:none}
.barra i{display:block;height:100%;width:0;background:var(--acento);transition:width .2s}
#galeria{display:grid;grid-template-columns:repeat(auto-fill,minmax(220px,1fr));gap:12px}
.tarjeta{background:var(--panel);border:1px solid var(--borde);border-radius:10px;overflow:hidden}
.tarjeta img{width:100%;display:block;cursor:zoom-in}
.tarjeta div{display:flex;justify-content:space-between;align-items:center;padding:8px 10px;font-size:12px;color:var(--suave)}
.tarjeta a{color:var(--acento);text-decoration:none;font-weight:600}
#grande{position:fixed;inset:0;background:rgba(0,0,0,.92);display:none;align-items:center;justify-content:center;z-index:9}
#grande img{max-width:96vw;max-height:94vh}
</style>
</head>
<body>
<main>
<h1>Linderman Imágenes</h1>

<div class="panel">
  <label for="texto">¿Qué imagen quiere? (en inglés sale mejor)</label>
  <textarea id="texto" placeholder="photo of a fisherman holding a big red snapper at the beach, golden hour"></textarea>

  <div class="fila">
    <div><label for="modelo">Modelo</label><select id="modelo"></select></div>
    <div><label for="tamano">Tamaño</label>
      <select id="tamano">
        <option value="1024x1024">Cuadrada (1:1)</option>
        <option value="832x1216">Vertical / Historia</option>
        <option value="1216x832">Horizontal</option>
        <option value="896x1152">Post vertical (4:5)</option>
      </select></div>
    <div><label for="cantidad">Cantidad</label>
      <select id="cantidad"><option>1</option><option>2</option><option>3</option><option>4</option></select></div>
  </div>

  <details>
    <summary>Más opciones</summary>
    <label for="lora">Estilo extra (LoRA)</label>
    <select id="lora"><option value="">Ninguno</option></select>
    <label for="fuerza">Fuerza del estilo extra (0.1 a 1.5)</label>
    <input id="fuerza" type="number" min="0.1" max="1.5" step="0.1" value="0.8">
    <label for="negativo">Lo que NO quiere que salga</label>
    <textarea id="negativo" style="min-height:60px"></textarea>
    <div class="fila">
      <div><label for="pasos">Pasos (más = más detalle, más lento)</label><input id="pasos" type="number" min="1" max="80"></div>
      <div><label for="cfg">Qué tanto obedece al texto</label><input id="cfg" type="number" min="1" max="15" step="0.5"></div>
      <div><label for="semilla">Semilla (vacío = al azar)</label><input id="semilla" type="number" min="0"></div>
    </div>
  </details>

  <button id="crear">Crear</button>
  <button id="cancelar">Cancelar</button>
  <div class="barra" id="barra"><i></i></div>
  <div id="estado">Cargando modelos…</div>
</div>

<div id="galeria"></div>
<div id="grande"><img alt=""></div>
</main>

<script>
const $ = id => document.getElementById(id);
const clientId = Math.random().toString(36).slice(2);
let modelos = [], trabajando = null;

// Ajustes recomendados por tipo de modelo
const PERFILES = {
  flux:   {nombre:"FLUX (máximo realismo, lento)", pasos:20, cfg:3.5, sampler:"euler", sched:"simple", pre:"", neg:"", maxCant:2},
  realvis:{nombre:"Fotos realistas (RealVisXL)", pasos:30, cfg:5, sampler:"dpmpp_2m_sde", sched:"karras", pre:"",
           neg:"lowres, worst quality, low quality, blurry, deformed, bad anatomy, bad hands, extra fingers, watermark, text, cartoon, 3d, painting, illustration"},
  noob:   {nombre:"Anime y personajes (NoobAI)", pasos:28, cfg:5, sampler:"euler_ancestral", sched:"normal",
           pre:"masterpiece, best quality, newest, absurdres, highres, ",
           neg:"worst quality, low quality, lowres, bad anatomy, bad hands, extra digits, fewer digits, watermark, signature, old, early, jpeg artifacts"},
  pony:   {nombre:"Pony", pasos:28, cfg:7, sampler:"euler_ancestral", sched:"normal", saltoClip:-2,
           pre:"score_9, score_8_up, score_7_up, ",
           neg:"score_4, score_5, score_6, worst quality, low quality, bad anatomy, bad hands, watermark"},
  sdxl:   {nombre:"", pasos:28, cfg:5.5, sampler:"dpmpp_2m", sched:"karras", pre:"",
           neg:"lowres, worst quality, low quality, blurry, bad anatomy, bad hands, watermark, text"}
};

function perfilDe(m){
  if(m.tipo === "flux") return "flux";
  const n = m.archivo.toLowerCase();
  if(n.includes("realvis")) return "realvis";
  if(n.includes("noob") || n.includes("illustrious") || n.includes("wai")) return "noob";
  if(n.includes("pony")) return "pony";
  return "sdxl";
}

async function lista(nodo, campo){
  try{
    const r = await fetch("/object_info/" + nodo);
    if(!r.ok) return [];
    const j = await r.json();
    return (j[nodo] && j[nodo].input.required[campo][0]) || [];
  }catch(e){ return []; }
}

async function cargar(){
  const ckpts = await lista("CheckpointLoaderSimple", "ckpt_name");
  const ggufs = (await lista("UnetLoaderGGUF", "unet_name")).filter(n => n.toLowerCase().includes("flux"));
  modelos = [...ggufs.map(a => ({archivo:a, tipo:"flux"})), ...ckpts.map(a => ({archivo:a, tipo:"sdxl"}))];
  // Orden: realista primero
  const orden = {realvis:0, flux:1, noob:2, pony:3, sdxl:4};
  modelos.sort((a,b) => orden[perfilDe(a)] - orden[perfilDe(b)]);
  $("modelo").innerHTML = modelos.map((m,i) =>
    `<option value="${i}">${PERFILES[perfilDe(m)].nombre || m.archivo.replace(/\.safetensors$/,"")}</option>`).join("");
  const loras = await lista("LoraLoader", "lora_name");
  $("lora").innerHTML = '<option value="">Ninguno</option>' + loras.map(l => `<option>${l}</option>`).join("");
  if(!modelos.length){ estado("No hay modelos. Revise en Kaggle que la celda de descarga terminó bien.", true); return; }
  aplicarPerfil();
  estado("Listo. Escriba qué quiere y apriete Crear.");
}

function aplicarPerfil(){
  const p = PERFILES[perfilDe(modelos[$("modelo").value])];
  $("pasos").value = p.pasos; $("cfg").value = p.cfg; $("negativo").value = p.neg;
  $("lora").disabled = $("fuerza").disabled = (p === PERFILES.flux);
}
$("modelo").onchange = aplicarPerfil;

function estado(t, err){ $("estado").textContent = t; $("estado").className = err ? "error" : ""; }

function flujoSDXL(m, p, d){
  const w = {
    "1": {class_type:"CheckpointLoaderSimple", inputs:{ckpt_name:m.archivo}},
    "9": {class_type:"VAELoader", inputs:{vae_name:"sdxl_vae_fp16_fix.safetensors"}},
    "4": {class_type:"EmptyLatentImage", inputs:{width:d.ancho, height:d.alto, batch_size:d.cant}},
    "5": {class_type:"KSampler", inputs:{seed:d.semilla, steps:d.pasos, cfg:d.cfg, sampler_name:p.sampler, scheduler:p.sched,
          denoise:1, model:["1",0], positive:["2",0], negative:["3",0], latent_image:["4",0]}},
    "6": {class_type:"VAEDecode", inputs:{samples:["5",0], vae:["9",0]}},
    "7": {class_type:"SaveImage", inputs:{filename_prefix:"linderman", images:["6",0]}}
  };
  let modelo = ["1",0], clip = ["1",1];
  if(d.lora){
    w["10"] = {class_type:"LoraLoader", inputs:{lora_name:d.lora, strength_model:d.fuerza, strength_clip:d.fuerza, model:modelo, clip:clip}};
    modelo = ["10",0]; clip = ["10",1];
  }
  if(p.saltoClip){
    w["11"] = {class_type:"CLIPSetLastLayer", inputs:{stop_at_clip_layer:p.saltoClip, clip:clip}};
    clip = ["11",0];
  }
  w["2"] = {class_type:"CLIPTextEncode", inputs:{text:p.pre + d.texto, clip:clip}};
  w["3"] = {class_type:"CLIPTextEncode", inputs:{text:d.neg, clip:clip}};
  w["5"].inputs.model = modelo;
  return w;
}

function flujoFLUX(m, d){
  return {
    "1": {class_type:"UnetLoaderGGUF", inputs:{unet_name:m.archivo}},
    "2": {class_type:"DualCLIPLoader", inputs:{clip_name1:"t5xxl_fp8_e4m3fn.safetensors", clip_name2:"clip_l.safetensors", type:"flux"}},
    "3": {class_type:"CLIPTextEncode", inputs:{text:d.texto, clip:["2",0]}},
    "4": {class_type:"FluxGuidance", inputs:{guidance:d.cfg, conditioning:["3",0]}},
    "12":{class_type:"ConditioningZeroOut", inputs:{conditioning:["3",0]}},
    "5": {class_type:"EmptySD3LatentImage", inputs:{width:d.ancho, height:d.alto, batch_size:d.cant}},
    "6": {class_type:"KSampler", inputs:{seed:d.semilla, steps:d.pasos, cfg:1, sampler_name:"euler", scheduler:"simple",
          denoise:1, model:["1",0], positive:["4",0], negative:["12",0], latent_image:["5",0]}},
    "8": {class_type:"VAELoader", inputs:{vae_name:"ae.safetensors"}},
    "9": {class_type:"VAEDecode", inputs:{samples:["6",0], vae:["8",0]}},
    "7": {class_type:"SaveImage", inputs:{filename_prefix:"linderman", images:["9",0]}}
  };
}

function barra(v){ $("barra").style.display = v === null ? "none" : "block"; $("barra").firstElementChild.style.width = (v||0) + "%"; }

// Progreso en vivo (si el túnel deja pasar el websocket; si no, igual funciona sin barra)
try{
  const ws = new WebSocket((location.protocol === "https:" ? "wss://" : "ws://") + location.host + "/ws?clientId=" + clientId);
  ws.onmessage = ev => {
    if(typeof ev.data !== "string" || !trabajando) return;
    const m = JSON.parse(ev.data);
    if(m.type === "progress" && m.data.prompt_id === trabajando.id){
      barra(100 * m.data.value / m.data.max);
      estado(`Creando… paso ${m.data.value} de ${m.data.max}`);
    }
  };
}catch(e){}

async function crear(){
  const texto = $("texto").value.trim();
  if(!texto){ estado("Escriba primero qué imagen quiere.", true); return; }
  const m = modelos[$("modelo").value], p = PERFILES[perfilDe(m)];
  const [ancho, alto] = $("tamano").value.split("x").map(Number);
  const d = {
    texto, ancho, alto,
    neg: $("negativo").value.trim(),
    cant: Math.min(Number($("cantidad").value), p.maxCant || 4),
    pasos: Math.max(1, Math.min(80, Number($("pasos").value) || p.pasos)),
    cfg: Number($("cfg").value) || p.cfg,
    semilla: $("semilla").value === "" ? Math.floor(Math.random() * 2**32) : Number($("semilla").value),
    lora: $("lora").value, fuerza: Number($("fuerza").value) || 0.8
  };
  const flujo = p === PERFILES.flux ? flujoFLUX(m, d) : flujoSDXL(m, p, d);

  $("crear").disabled = true; $("cancelar").style.display = "block"; barra(0);
  estado("Enviando… (la primera imagen de cada modelo tarda más porque lo carga)");
  const inicio = Date.now();
  try{
    const r = await fetch("/prompt", {method:"POST", headers:{"Content-Type":"application/json"},
                                      body: JSON.stringify({prompt: flujo, client_id: clientId})});
    const j = await r.json();
    if(!r.ok || j.error){
      const detalle = j.node_errors ? Object.values(j.node_errors).map(n => n.errors.map(e => e.details || e.message).join("; ")).join(" | ") : "";
      throw new Error((j.error && j.error.message || "Error") + (detalle ? ": " + detalle : ""));
    }
    trabajando = {id: j.prompt_id};
    const salida = await esperar(j.prompt_id);
    const imgs = Object.values(salida.outputs || {}).flatMap(o => o.images || []);
    if(!imgs.length) throw new Error("No salió ninguna imagen (se canceló o hubo un error en Kaggle).");
    imgs.forEach(im => agregar(im, d.semilla));
    estado(`Listo en ${Math.round((Date.now() - inicio)/1000)} s. Semilla ${d.semilla}.`);
  }catch(e){
    estado(e.message, true);
  }finally{
    trabajando = null; $("crear").disabled = false; $("cancelar").style.display = "none"; barra(null);
  }
}

async function esperar(id){
  while(true){
    await new Promise(r => setTimeout(r, 1500));
    let j;
    try{ j = await (await fetch("/history/" + id)).json(); }catch(e){ continue; } // el túnel a veces corta una petición
    const h = j[id];
    if(!h) continue;
    if(h.status && h.status.status_str === "error"){
      if((h.status.messages || []).some(x => x[0] === "execution_interrupted")) throw new Error("Cancelado.");
      const msg = (h.status.messages || []).find(x => x[0] === "execution_error");
      throw new Error("Error en Kaggle: " + (msg ? msg[1].exception_message : "revise la celda de Kaggle"));
    }
    if(h.status && h.status.completed) return h;
    if(h.outputs && Object.keys(h.outputs).length) return h;
  }
}

function agregar(im, semilla){
  const url = `/view?filename=${encodeURIComponent(im.filename)}&subfolder=${encodeURIComponent(im.subfolder)}&type=${im.type}`;
  const t = document.createElement("div");
  t.className = "tarjeta";
  t.innerHTML = `<img src="${url}" alt="" loading="lazy"><div><span>Semilla ${semilla}</span><a href="${url}" download="${im.filename}">Bajar</a></div>`;
  t.querySelector("img").onclick = () => { $("grande").firstElementChild.src = url; $("grande").style.display = "flex"; };
  $("galeria").prepend(t);
}

$("grande").onclick = () => $("grande").style.display = "none";
$("crear").onclick = crear;
$("cancelar").onclick = () => fetch("/interrupt", {method:"POST"});
$("texto").addEventListener("keydown", e => { if(e.key === "Enter" && (e.ctrlKey || e.metaKey)) crear(); });
cargar();
</script>
</body>
</html>
'''


In [ ]:
# ===== 1. Revisar la máquina e instalar ComfyUI =====
import os, sys, re, json, time, shutil, subprocess, urllib.request

class Alto(Exception):
    pass

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        raise Alto("Falló este paso: " + cmd)
    return r.stdout

try:
    gpu = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader").strip().splitlines()[0]
except Alto:
    raise Alto("No hay tarjeta. Panel derecho → Accelerator → GPU T4 x2, y vuelva a darle Run All.")
print("✅ Tarjeta:", gpu)
try:
    urllib.request.urlopen("https://huggingface.co", timeout=20)
except Exception:
    raise Alto("Sin internet. Panel derecho → Internet: On (necesita el teléfono verificado en Kaggle).")
print("✅ Internet")

BASE = max([d for d in ["/kaggle/temp", "/tmp"] if os.path.isdir(d)], key=lambda d: shutil.disk_usage(d).free)
COMFY = BASE + "/ComfyUI"
SALIDA = "/kaggle/working/imagenes"
os.makedirs(SALIDA, exist_ok=True)

if not os.path.exists(COMFY + "/main.py"):
    try:
        tag = json.load(urllib.request.urlopen("https://api.github.com/repos/comfyanonymous/ComfyUI/releases/latest", timeout=20))["tag_name"]
    except Exception:
        tag = "master"
    sh(f"git clone -q --depth 1 --branch {tag} https://github.com/comfyanonymous/ComfyUI {COMFY}")
    print("✅ ComfyUI", tag)
GGUF = COMFY + "/custom_nodes/ComfyUI-GGUF"
if not os.path.exists(GGUF):
    sh(f"git clone -q --depth 1 https://github.com/city96/ComfyUI-GGUF {GGUF}")

# Dependencias, sin tocar el torch que ya trae Kaggle (cambiarlo rompe la tarjeta)
reqs = [l for l in open(COMFY + "/requirements.txt").read().splitlines()
        if l.strip() and not l.startswith("#") and not re.match(r"^(torch|torchvision|torchaudio)\s*($|[<>=!~;])", l.strip())]
reqs += [l for l in open(GGUF + "/requirements.txt").read().splitlines() if l.strip() and not l.startswith("#")]
open(BASE + "/reqs.txt", "w").write("\n".join(reqs))
sh(f"{sys.executable} -m pip install -q --disable-pip-version-check -r {BASE}/reqs.txt")
print("✅ Dependencias")

# La página sencilla (se abre en  .../app)
APP_DIR = COMFY + "/custom_nodes/linderman_app"
os.makedirs(APP_DIR, exist_ok=True)
open(APP_DIR + "/app.html", "w", encoding="utf-8").write(APP_HTML)
open(APP_DIR + "/__init__.py", "w").write('''import os
from aiohttp import web
from server import PromptServer

_HTML = os.path.join(os.path.dirname(os.path.realpath(__file__)), "app.html")

@PromptServer.instance.routes.get("/app")
async def linderman_app(request):
    return web.FileResponse(_HTML, headers={"Cache-Control": "no-store"})

NODE_CLASS_MAPPINGS = {}
NODE_DISPLAY_NAME_MAPPINGS = {}
''')
print("✅ Página lista")


In [ ]:
# ===== 2. Bajar los modelos =====
from concurrent.futures import ThreadPoolExecutor
import struct

HF = "https://huggingface.co/"
M = COMFY + "/models"
lista = [("vae", "sdxl_vae_fp16_fix.safetensors", HF + "madebyollin/sdxl-vae-fp16-fix/resolve/main/sdxl.vae.safetensors", 0.4)]
if MODELO_REALISTA:
    lista.append(("checkpoints", "RealVisXL_V5.0_fp16.safetensors", HF + "SG161222/RealVisXL_V5.0/resolve/main/RealVisXL_V5.0_fp16.safetensors", 7))
if MODELO_ANIME:
    lista.append(("checkpoints", "NoobAI-XL-v1.1.safetensors", HF + "Laxhar/noobai-XL-1.1/resolve/main/NoobAI-XL-v1.1.safetensors", 7))
if MODELO_FLUX:
    lista += [("unet", "flux1-dev-Q8_0.gguf", HF + "city96/FLUX.1-dev-gguf/resolve/main/flux1-dev-Q8_0.gguf", 12.7),
              ("clip", "t5xxl_fp8_e4m3fn.safetensors", HF + "comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors", 4.9),
              ("clip", "clip_l.safetensors", HF + "comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors", 0.3),
              ("vae", "ae.safetensors", HF + "Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors", 0.4)]

# Civitai: se pregunta qué es cada número antes de bajarlo
if CIVITAI:
    try:
        from kaggle_secrets import UserSecretsClient
        TOKEN = UserSecretsClient().get_secret("CIVITAI_TOKEN")
    except Exception:
        raise Alto("Falta el secreto CIVITAI_TOKEN (Add-ons → Secrets) o no tiene el cuadrito activado para este cuaderno.")
    for vid in CIVITAI:
        try:
            info = json.load(urllib.request.urlopen(urllib.request.Request(
                f"https://civitai.com/api/v1/model-versions/{vid}", headers={"Authorization": "Bearer " + TOKEN}), timeout=30))
        except Exception as e:
            print(f"⚠️ Civitai {vid}: no se encontró ({e}). Lo salto."); continue
        tipo, base = info["model"]["type"], info.get("baseModel", "")
        carpeta = {"Checkpoint": "checkpoints", "LORA": "loras", "LoCon": "loras"}.get(tipo)
        if not carpeta:
            print(f"⚠️ Civitai {vid}: es tipo {tipo}, la página solo usa modelos y LoRAs. Lo salto."); continue
        if not any(x in base for x in ["SDXL", "Pony", "Illustrious", "NoobAI"]):
            print(f"⚠️ Civitai {vid}: es para {base}; la página solo usa SDXL, Pony o Illustrious. Lo salto."); continue
        arch = [f for f in info["files"] if f["name"].endswith(".safetensors")]
        if not arch:
            print(f"⚠️ Civitai {vid}: no trae archivo .safetensors (los .ckpt pueden traer virus). Lo salto."); continue
        f = next((x for x in arch if x.get("primary")), arch[0])
        sep = "&" if "?" in f["downloadUrl"] else "?"
        lista.append((carpeta, f["name"], f["downloadUrl"] + sep + "token=" + TOKEN, f["sizeKB"] / 1024 / 1024))
        print(f"• Civitai {vid}: {info['model']['name']} ({tipo}, {base})")

falta = sum(t for c, n, u, t in lista if not os.path.exists(f"{M}/{c}/{n}"))
libre = shutil.disk_usage(BASE).free / 1e9
if falta * 1.07 + 3 > libre:
    raise Alto(f"No cabe: faltan {falta:.0f} GB y hay {libre:.0f} GB libres. Apague MODELO_FLUX o algún otro en AJUSTES.")

def archivo_bueno(ruta):
    # Revisa que sea un modelo de verdad y no una página de error
    with open(ruta, "rb") as fh:
        ini = fh.read(8)
        if ruta.endswith(".gguf"):
            return ini[:4] == b"GGUF"
        n = struct.unpack("<Q", ini)[0]
        if n > 100_000_000:
            return False
        json.loads(fh.read(n))
    return True

def bajar(item):
    carpeta, nombre, url, gb = item
    destino = f"{M}/{carpeta}/{nombre}"
    if os.path.exists(destino):
        return f"✅ {nombre} (ya estaba)"
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    inicio = time.time()
    for intento in range(3):
        r = subprocess.run(["curl", "-L", "--fail", "-s", "-S", "--retry", "5", "--retry-delay", "5",
                            "-C", "-", "-o", destino + ".part", url], capture_output=True, text=True)
        if r.returncode == 0:
            break
        time.sleep(5)
    else:
        return f"❌ {nombre}: no bajó ({r.stderr.strip()[-200:]})"
    try:
        ok = archivo_bueno(destino + ".part")
    except Exception:
        ok = False
    if not ok:
        os.remove(destino + ".part")
        return f"❌ {nombre}: llegó dañado (si es de Civitai, revise el token)"
    os.rename(destino + ".part", destino)
    return f"✅ {nombre}: {os.path.getsize(destino)/1e9:.1f} GB en {time.time()-inicio:.0f} s"

print(f"Bajando {len(lista)} archivos (~{falta:.0f} GB)…")
with ThreadPoolExecutor(3) as ex:
    resultados = list(ex.map(bajar, lista))
print("\n".join(resultados))
if any(r.startswith("❌") for r in resultados):
    raise Alto("Algún archivo no bajó. Vuelva a correr esta celda: sigue donde se quedó.")


In [ ]:
# ===== 3. Prender el generador y sacar el link =====
from IPython.display import display, HTML

def cola(ruta, n=40):
    try:
        return "".join(open(ruta, errors="replace").readlines()[-n:])
    except Exception:
        return ""

def prender_comfy():
    subprocess.run("pkill -f 'ComfyUI/main.py'", shell=True)
    log = open(BASE + "/comfy.log", "w")
    p = subprocess.Popen([sys.executable, COMFY + "/main.py", "--listen", "127.0.0.1", "--port", "8188",
                          "--output-directory", SALIDA, "--preview-method", "none", "--disable-auto-launch"],
                         cwd=COMFY, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(300):
        if p.poll() is not None:
            print(cola(BASE + "/comfy.log"))
            raise Alto("ComfyUI no arrancó (el error está arriba). Mándele esa parte a Linderman.")
        try:
            urllib.request.urlopen("http://127.0.0.1:8188/app", timeout=3)
            return p
        except Exception:
            time.sleep(2)
    raise Alto("ComfyUI tardó demasiado en arrancar.\n" + cola(BASE + "/comfy.log"))

def prender_tunel():
    cf = BASE + "/cloudflared"
    if not os.path.exists(cf):
        sh(f"curl -L --fail -s -o {cf} https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x {cf}")
    log = BASE + "/tunel.log"
    p = subprocess.Popen([cf, "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8188"],
                         stdout=open(log, "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", cola(log, 200))
        if m:
            return p, m.group(0)
        if p.poll() is not None:
            break
        time.sleep(2)
    raise Alto("No salió el link.\n" + cola(log))

def mostrar(url):
    display(HTML(f'<a href="{url}/app" target="_blank" style="display:inline-block;background:#3fb27f;color:#06130d;'
                 f'padding:14px 22px;border-radius:10px;font:700 20px sans-serif;text-decoration:none">ABRIR GENERADOR</a>'
                 f'<p style="font-size:15px">{url}/app &nbsp;·&nbsp; ComfyUI completo: {url}</p>'))

comfy = prender_comfy()
print("✅ ComfyUI prendido")
tunel, url = prender_tunel()
time.sleep(5)  # el link nuevo tarda unos segundos en responder
mostrar(url)
print("Deje esta celda corriendo. Para apagar: botón ■ (Stop) arriba.")

try:
    while True:
        time.sleep(30)
        if comfy.poll() is not None:
            print(cola(BASE + "/comfy.log"))
            print("⚠️ ComfyUI se cayó (arriba está por qué). Lo vuelvo a prender…")
            comfy = prender_comfy()
        if tunel.poll() is not None:
            print("⚠️ Se cortó el link. Saco uno nuevo:")
            tunel, url = prender_tunel()
            mostrar(url)
except KeyboardInterrupt:
    tunel.terminate(); comfy.terminate()
    print("Apagado. Sus imágenes están en Output → imagenes.")


In [ ]:
# ===== 4. (Opcional) Juntar todas las imágenes en un .zip para bajarlas de una vez =====
# Primero pare la celda 3 con ■ y luego corra esta.
from IPython.display import FileLink
shutil.make_archive("/kaggle/working/imagenes_linderman", "zip", SALIDA)
FileLink("imagenes_linderman.zip")
